### Control Matching
This notebook focuses on refining matched pairs of junior award-winning authors (treated group) with control authors in academic conferences. Key steps include:

- Loading and processing data: Matched pairs, author profiles, and junior authors from CSV files.
- Augmenting profiles: Calculating career age (award year minus first publication year) for all authors.
- Splitting data: Identifying clean juniors (treatment=1) and controls (treatment=0 with career age >5).
- Building a control pool: Extracting authors from award papers, excluding known juniors and those with insufficient career age.
- Re-matching: Using NearestNeighbors (Euclidean distance) on encoded features (conference, author position, award year) to pair juniors with closest controls where distance <1.5.
- **1-to-1 matching enforced**: each control author is used at most once.
- Saving results: Exporting the new matched pairs to a CSV file.

In [1]:
import pandas as pd, ast, json, numpy as np
from pathlib import Path

ROOT = Path("..")
MATCHED = ROOT / "data" / "matched"
PROFILES = ROOT / "data" / "profiles"

pairs    = pd.read_csv(MATCHED / "matched_pairs.csv")
profiles = pd.read_csv(PROFILES / "all_matched_profiles.csv")
juniors  = pd.read_csv(MATCHED / "junior_authors_all_conferences.csv")

def first_pub_year(counts_raw):
    try:
        entries = ast.literal_eval(counts_raw)
        years = [e["year"] for e in entries if e.get("works_count", 0) > 0]
        return min(years) if years else None
    except:
        return None

# Add career age to ALL profiles
profiles_aug = profiles.copy()
profiles_aug["first_pub"] = profiles_aug["counts_by_year"].apply(first_pub_year)
profiles_aug["career_age"] = profiles_aug["award_year"] - profiles_aug["first_pub"]

# Split clean
clean_juniors = profiles_aug[profiles_aug["treatment"] == 1].copy()
clean_controls = profiles_aug[
    (profiles_aug["treatment"] == 0) &
    (profiles_aug["career_age"] > 5)
].copy()

print(f"Clean juniors:  {len(clean_juniors)}")
print(f"Clean controls: {len(clean_controls)}")
print(f"Removed contaminated controls: {(profiles_aug['treatment']==0).sum() - len(clean_controls)}")


Clean juniors:  603
Clean controls: 464
Removed contaminated controls: 139


In [2]:
from sklearn.neighbors import NearestNeighbors
import json as js
import requests, time

MAILTO = "shaheryar.4822@student.uu.se"

all_awards = pd.read_csv(MATCHED / "huang_matched_openalex.csv")

def parse_authorships(raw):
    if isinstance(raw, list): return raw
    if pd.isna(raw): return []
    try: return js.loads(raw.replace("'", '"'))
    except:
        try:
            import ast as ast2
            return ast2.literal_eval(raw)
        except: return []

all_authors = []
for _, row in all_awards.iterrows():
    authorships = parse_authorships(row["authorships"])
    for pos, a in enumerate(authorships[:5], 1):
        author = a.get("author", {})
        all_authors.append({
            "author_id":       author.get("id"),
            "author_name":     author.get("display_name"),
            "conference":      row["conference"],
            "award_year":      row["year"],
            "author_position": pos,
        })

all_df = pd.DataFrame(all_authors)

# Fetch career age for authors NOT already in profiles_aug
def fetch_first_pub(author_id):
    aid = author_id.split("/")[-1]
    url = "https://api.openalex.org/works"
    params = {
        "filter":   f"author.id:{aid}",
        "sort":     "publication_year:asc",
        "per-page": 1,
        "select":   "publication_year",
        "mailto":   MAILTO,
    }
    try:
        r = requests.get(url, params=params, timeout=10)
        results = r.json().get("results", [])
        if results:
            return results[0].get("publication_year")
    except:
        pass
    return None

career_lookup = profiles_aug.drop_duplicates("author_id").set_index("author_id")["career_age"].to_dict()

known_ids = set(career_lookup.keys())
missing = all_df[~all_df["author_id"].isin(known_ids)].drop_duplicates("author_id").dropna(subset=["author_id"])

print(f"Authors needing career age fetch: {len(missing)}")

for i, (_, row) in enumerate(missing.iterrows()):
    first_pub = fetch_first_pub(row["author_id"])
    if first_pub:
        career_lookup[row["author_id"]] = row["award_year"] - first_pub
    if i % 100 == 0:
        print(f"  fetched {i}/{len(missing)}...")
    time.sleep(0.15)

all_df["career_age"] = all_df["author_id"].map(career_lookup)
print(f"Career age coverage: {all_df['career_age'].notna().sum()} / {len(all_df)}")

# Controls = NOT a known junior AND career age > 5
junior_ids = set(juniors["author_id"])
controls_pool = all_df[
    (~all_df["author_id"].isin(junior_ids)) &
    (all_df["career_age"] > 5)
].copy()

print(f"Clean control pool: {len(controls_pool)}")

# Encode features for matching
features = ["conference", "author_position", "award_year"]
combined = pd.concat([juniors.assign(_src='treat'), controls_pool.assign(_src='ctrl')], ignore_index=True)
for f in features:
    combined[f + '_enc'] = pd.Categorical(combined[f]).codes

treat_df = combined[combined['_src'] == 'treat'].reset_index(drop=True)
ctrl_df  = combined[combined['_src'] == 'ctrl'].reset_index(drop=True)

treat_feat = treat_df[[f + '_enc' for f in features]].to_numpy()
ctrl_feat  = ctrl_df[[f + '_enc' for f in features]].to_numpy()

# 1-to-1 matching: each control used at most once
nn = NearestNeighbors(n_neighbors=1, metric="euclidean")
nn.fit(ctrl_feat)
dists, ctrl_idxs = nn.kneighbors(treat_feat)

good = dists[:, 0] < 1.5
used_ctrl = set()
new_pairs = []

for i in np.where(good)[0]:
    ctrl_idx = ctrl_idxs[i, 0]
    ctrl_id  = ctrl_df.iloc[ctrl_idx]["author_id"]

    if ctrl_id in used_ctrl:
        continue  # skip already-assigned control

    used_ctrl.add(ctrl_id)
    j = treat_df.iloc[i]
    c = ctrl_df.iloc[ctrl_idx]
    new_pairs.append({
        "treated_id":   j["author_id"],
        "treated_name": j["author_name"],
        "control_id":   c["author_id"],
        "control_name": c["author_name"],
        "conference":   j["conference"],
        "award_year":   j["award_year"],
        "treat_pos":    j["author_position"],
        "ctrl_pos":     c["author_position"],
        "match_dist":   float(dists[i, 0]),
    })

new_matched = pd.DataFrame(new_pairs)
new_matched.to_csv(MATCHED / "matched_pairs_clean.csv", index=False)

print(f"\nNew matched pairs: {len(new_matched)}")
print(f"Unique treated:    {new_matched['treated_id'].nunique()}")
print(f"Unique controls:   {new_matched['control_id'].nunique()}")
print(f"Avg distance:      {new_matched['match_dist'].mean():.3f}")


Authors needing career age fetch: 1727
  fetched 0/1727...
  fetched 100/1727...
  fetched 200/1727...
  fetched 300/1727...
  fetched 400/1727...
  fetched 500/1727...
  fetched 600/1727...
  fetched 700/1727...
  fetched 800/1727...
  fetched 900/1727...
  fetched 1000/1727...
  fetched 1100/1727...
  fetched 1200/1727...
  fetched 1300/1727...
  fetched 1400/1727...
  fetched 1500/1727...
  fetched 1600/1727...
  fetched 1700/1727...
Career age coverage: 2925 / 2964
Clean control pool: 2121

New matched pairs: 345
Unique treated:    345
Unique controls:   345
Avg distance:      0.455
